<a href="https://colab.research.google.com/github/ProductPriceTrackerOrg/data-science/blob/main/notebooks/product-matching/01_model_building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Product Matching Model - Training a Multi-Modal Product Matching Model**

## **Import Required Libraries**

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet50
from sentence_transformers import SentenceTransformer
from PIL import Image
import os
import time
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
!git clone "https://github.com/ProductPriceTrackerOrg/data-science.git"

Cloning into 'data-science'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 116 (delta 27), reused 79 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 7.73 MiB | 7.86 MiB/s, done.
Resolving deltas: 100% (27/27), done.


## **2: Custom Dataset Class**

In [ ]:
class ProductMatchingDataset(Dataset):
    def __init__(self, dataframe, text_embeddings_dict, image_transform=None, mode='multimodal'):
        """
        Args:
            dataframe: pandas DataFrame with product pairs
            text_embeddings_dict: Dictionary mapping text to pre-computed embeddings
            image_transform: torchvision transforms for images
            mode: 'text_only' or 'multimodal'
        """
        self.df = dataframe.reset_index(drop=True)
        self.text_embeddings = text_embeddings_dict
        self.image_transform = image_transform
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Get pre-computed text embeddings
        text1_key = f"{row['name1']} {row['description1']}"
        text2_key = f"{row['name2']} {row['description2']}"

        text_emb1 = self.text_embeddings[text1_key]
        text_emb2 = self.text_embeddings[text2_key]

        label = torch.tensor(row['match'], dtype=torch.float32)

        if self.mode == 'text_only':
            return {
                'text1': torch.tensor(text_emb1, dtype=torch.float32),
                'text2': torch.tensor(text_emb2, dtype=torch.float32),
                'label': label
            }

        elif self.mode == 'multimodal':
            # Load and process images
            try:
                image1 = Image.open(row['image1_path']).convert('RGB')
                image2 = Image.open(row['image2_path']).convert('RGB')

                if self.image_transform:
                    image1 = self.image_transform(image1)
                    image2 = self.image_transform(image2)

                return {
                    'text1': torch.tensor(text_emb1, dtype=torch.float32),
                    'text2': torch.tensor(text_emb2, dtype=torch.float32),
                    'image1': image1,
                    'image2': image2,
                    'label': label
                }
            except Exception as e:
                print(f"Error loading images at index {idx}: {e}")
                # Return dummy images if loading fails
                dummy_image = torch.zeros(3, 224, 224)
                return {
                    'text1': torch.tensor(text_emb1, dtype=torch.float32),
                    'text2': torch.tensor(text_emb2, dtype=torch.float32),
                    'image1': dummy_image,
                    'image2': dummy_image,
                    'label': label
                }

## **3: Data Loading and Preprocessing**

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('/content/data-science/data/raw/product-matching/amazon_train_data.csv')
    print(f"Dataset loaded successfully. Shape: {df.shape}")
    print("\nDataset columns:", df.columns.tolist())
    print("\nFirst few rows:")
    display(df.head())
except FileNotFoundError:
    print("Creating dummy dataset for demonstration...")
    # Create dummy dataset for demonstration
    np.random.seed(42)
    n_samples = 1000

    df = pd.DataFrame({
        'name1': [f'Product A {i}' for i in range(n_samples)],
        'description1': [f'Description for product A {i}' for i in range(n_samples)],
        'image1_path': [f'images/product_a_{i}.jpg' for i in range(n_samples)],
        'id1': range(n_samples),
        'name2': [f'Product B {i}' for i in range(n_samples)],
        'description2': [f'Description for product B {i}' for i in range(n_samples)],
        'image2_path': [f'images/product_b_{i}.jpg' for i in range(n_samples)],
        'id2': range(n_samples, 2*n_samples),
        'match': np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3])
    })
    print(f"Dummy dataset created. Shape: {df.shape}")

# Check class distribution
print(f"\nClass distribution:")
print(df['match'].value_counts())



Dataset loaded successfully. Shape: (1600, 19)

Dataset columns: ['name1', 'short_description1', 'long_description1', 'specification1', 'image1', 'price1', 'id1', 'name2', 'short_description2', 'long_description2', 'specification2', 'image2', 'price2', 'id2', 'match', 'category', 'match_type', 'image_url1', 'image_url2']

First few rows:


,name1,short_description1,long_description1,specification1,image1,price1,id1,name2,short_description2,long_description2,specification2,image2,price2,id2,match,category,match_type,image_url1,image_url2
0,8pcs Flamingo Pattern Ice Cream Paper Bowls Yo...,&lt;strong&gt;Description&lt;/strong&gt;&lt;br...,Safe paper material will not cause the softnes...,"[{""key"": ""Assembled Product Dimensions (L x W ...",6,6.99,https://walmart.com/ip/8pcs-Flamingo-Pattern-I...,Hemoton 8pcs Paper Ice Cream Cups Dessert Bowl...,"Perfect for snacks, desserts, ice cream, or an...",Description Our disposable dessert cups are ma...,"[{""key"": ""Brand"", ""value"": ""Hemoton""}, {""key"":...",1,7.99,https://www.amazon.com/dp/B09Y5J2ZJB,1,6_household,match,"[""https://i5.walmartimages.com/asr/b3142085-18...","[""https://m.media-amazon.com/images/I/61jLqYHc..."
1,Arm & Hammer Moisture Absorber Refills Fragran...,The Arm & Hammer 2 Pack Moisture Absorber and ...,About this item Arm &amp; Hammer Moisture Abso...,"[{""key"": ""Brand"", ""value"": ""Arm & Hammer""}]",7,14.99,https://walmart.com/ip/Arm-Hammer-Moisture-Abs...,Arm & Hammer Fragrance Free Disposable Moistur...,Arm & Hammer Disposable Moisture Absorber and ...,The Arm & Hammer Disposable Moisture Absorber ...,[],1,NaN,https://www.amazon.com/dp/B01KYZ13TM,0,6_household,close_nonmatch,"[""https://i5.walmartimages.com/asr/b7678bb4-7b...","[""https://m.media-amazon.com/images/I/81Bexk3J..."
2,Bissell Double Action Brush Roll for Select Up...,Brand new Bissell Double Action Brush Roll for...,Bissell Double Action Brush Roll for Select Up...,"[{""key"": ""Brand"", ""value"": ""BISSELL""}]",2,24.32,https://walmart.com/ip/Bissell-Double-Action-B...,Replacement Part For Bissell Gray Brushroll fo...,compare to part number 1611230 Fits In Models ...,"Fits In Models # 14522, 3198A, 3196, 3198, 319...","[{""key"": ""Manufacturer"", ""value"": ""Top Vacuum ...",1,NaN,https://www.amazon.com/dp/B08XJT1CG8,0,6_household,close_nonmatch,"[""https://i5.walmartimages.com/asr/b4a20761-d0...","[""https://m.media-amazon.com/images/I/31Ah9vIp..."
3,"BISSELL MYair Plus, for rooms up to 100 sq. ft...",Enjoy Simply Clean AirÃ¢â€žÂ¢ from a brand you...,Purify Air in Small Rooms. Cleans air in mediu...,"[{""key"": ""Features"", ""value"": ""Filter Change I...",7,123.59,https://walmart.com/ip/BISSELL-MYair-Plus-for-...,BISSELL MYair Pro Air Purifier with HEPA Filte...,Every Purchase Saves Pets. BISSELL proudly sup...,Get Simply Clean AirÃ¢â€žÂ¢ from the brand you...,"[{""key"": ""Color"", ""value"": ""White""}, {""key"": ""...",1,102.99,https://www.amazon.com/dp/B093ZDWCQ5,0,6_household,close_nonmatch,"[""https://i5.walmartimages.com/asr/706bfb2b-0d...","[""https://m.media-amazon.com/images/I/71q0sCxm..."
4,Clorox Hideaway Bowl Brush and Holder with Base,Clorox takes the work out of cleaning your com...,Antimicrobial treatment on brush bristles add ...,"[{""key"": ""Brand"", ""value"": ""Clorox""}, {""key"": ...",6,7.98,https://walmart.com/ip/Clorox-Hideaway-Bowl-Br...,Clorox Under-Rim Toilet Bowl Brush with Corner...,TOILET BOWL BRUSH AND HOLDER SET: Includes an ...,Keep your bathroom clean and organized with th...,[],1,NaN,https://www.amazon.com/dp/B00R55CIRQ,1,6_household,match,"[""https://i5.walmartimages.com/asr/73956f8d-b2...","[""https://m.media-amazon.com/images/I/61GcSaqG..."



Class distribution:
match
0    1184
1     416
Name: count, dtype: int64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600 entries, 0 to 1599
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   name1               1600 non-null   object 
 1   short_description1  1598 non-null   object 
 2   long_description1   1598 non-null   object 
 3   specification1      1600 non-null   object 
 4   image1              1600 non-null   int64  
 5   price1              1384 non-null   float64
 6   id1                 1600 non-null   object 
 7   name2               1600 non-null   object 
 8   short_description2  1503 non-null   object 
 9   long_description2   1201 non-null   object 
 10  specification2      1600 non-null   object 
 11  image2              1600 non-null   int64  
 12  price2              1246 non-null   float64
 13  id2                 1600 non-null   object 
 14  match               1600 non-null   int64  
 15  category            1600 non-null   object 
 16  match_

In [ ]:
import ast

def transform_df(df: pd.DataFrame) -> pd.DataFrame:
    # Convert strings to lists if needed
    df["image_url1"] = df["image_url1"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    df["image_url2"] = df["image_url2"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # Build new dataframe
    new_df = pd.DataFrame({
        "name1": df["name1"],
        "description1": df["long_description1"],
        "image1_path": [
            urls[idx - 1] if isinstance(urls, list) and 0 <= (idx - 1) < len(urls) else None
            for urls, idx in zip(df["image_url1"], df["image1"])
        ],
        "id1": df["id1"],
        "name2": df["name2"],
        "description2": df["long_description2"],
        "image2_path": [
            urls[idx - 1] if isinstance(urls, list) and 0 <= (idx - 1) < len(urls) else None
            for urls, idx in zip(df["image_url2"], df["image2"])
        ],
        "id2": df["id2"],
        "match": df["match"]
    })

    return new_df

df_new = transform_df(df)

In [ ]:
display(df_new.head())
df = df_new

,name1,description1,image1_path,id1,name2,description2,image2_path,id2,match
0,8pcs Flamingo Pattern Ice Cream Paper Bowls Yo...,Safe paper material will not cause the softnes...,https://i5.walmartimages.com/asr/bf1456c9-7da2...,https://walmart.com/ip/8pcs-Flamingo-Pattern-I...,Hemoton 8pcs Paper Ice Cream Cups Dessert Bowl...,Description Our disposable dessert cups are ma...,https://m.media-amazon.com/images/I/61jLqYHcj9...,https://www.amazon.com/dp/B09Y5J2ZJB,1
1,Arm & Hammer Moisture Absorber Refills Fragran...,About this item Arm &amp; Hammer Moisture Abso...,https://i5.walmartimages.com/asr/43cd1ae7-95fa...,https://walmart.com/ip/Arm-Hammer-Moisture-Abs...,Arm & Hammer Fragrance Free Disposable Moistur...,The Arm & Hammer Disposable Moisture Absorber ...,https://m.media-amazon.com/images/I/81Bexk3JlM...,https://www.amazon.com/dp/B01KYZ13TM,0
2,Bissell Double Action Brush Roll for Select Up...,Bissell Double Action Brush Roll for Select Up...,https://i5.walmartimages.com/asr/cd821a78-5ef4...,https://walmart.com/ip/Bissell-Double-Action-B...,Replacement Part For Bissell Gray Brushroll fo...,"Fits In Models # 14522, 3198A, 3196, 3198, 319...",https://m.media-amazon.com/images/I/31Ah9vIpji...,https://www.amazon.com/dp/B08XJT1CG8,0
3,"BISSELL MYair Plus, for rooms up to 100 sq. ft...",Purify Air in Small Rooms. Cleans air in mediu...,https://i5.walmartimages.com/asr/9e004637-269e...,https://walmart.com/ip/BISSELL-MYair-Plus-for-...,BISSELL MYair Pro Air Purifier with HEPA Filte...,Get Simply Clean AirÃ¢â€žÂ¢ from the brand you...,https://m.media-amazon.com/images/I/71q0sCxmVc...,https://www.amazon.com/dp/B093ZDWCQ5,0
4,Clorox Hideaway Bowl Brush and Holder with Base,Antimicrobial treatment on brush bristles add ...,https://i5.walmartimages.com/asr/ac0a7a05-68a6...,https://walmart.com/ip/Clorox-Hideaway-Bowl-Br...,Clorox Under-Rim Toilet Bowl Brush with Corner...,Keep your bathroom clean and organized with th...,https://m.media-amazon.com/images/I/61GcSaqGBi...,https://www.amazon.com/dp/B00R55CIRQ,1


In [ ]:
# Split the data
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['match'])
val_df, test_df = train_test_split(temp_df, test_size=0.4, random_state=42, stratify=temp_df['match'])

print(f"\nDataset splits:")
print(f"Train: {len(train_df)} samples")
print(f"Validation: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")


Dataset splits:
Train: 1280 samples
Validation: 192 samples
Test: 128 samples


## **4: Initialize Text Encoder and Image Transforms**

In [ ]:
# Initialize text encoder
print("Loading SentenceTransformer model...")
text_encoder = SentenceTransformer('all-MiniLM-L6-v2')
print("Text encoder loaded successfully!")

# Define image transforms
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

print("Image transforms defined!")

# Get text embedding dimension
sample_text = "sample text"
text_dim = len(text_encoder.encode(sample_text))
print(f"Text embedding dimension: {text_dim}")

Loading SentenceTransformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Text encoder loaded successfully!
Image transforms defined!
Text embedding dimension: 384


In [ ]:
# Pre-compute all text embeddings for efficiency
print("Pre-computing all text embeddings...")
print("This is a one-time process that will significantly speed up training...")

# Collect all unique texts
all_texts = []
for _, row in df.iterrows():
    text1 = f"{row['name1']} {row['description1']}"
    text2 = f"{row['name2']} {row['description2']}"
    all_texts.extend([text1, text2])

# Get unique texts to avoid redundant computation
unique_texts = list(set(all_texts))
print(f"Total unique text combinations: {len(unique_texts)}")

# Pre-compute embeddings for all unique texts
print("Computing embeddings... This may take a few minutes...")
text_to_embedding = {}

# Process in batches for better efficiency
batch_size = 64
for i in range(0, len(unique_texts), batch_size):
    batch_texts = unique_texts[i:i+batch_size]
    batch_embeddings = text_encoder.encode(batch_texts, convert_to_tensor=False)

    for text, embedding in zip(batch_texts, batch_embeddings):
        text_to_embedding[text] = embedding

    if (i // batch_size + 1) % 10 == 0:
        print(f"Processed {i + len(batch_texts)}/{len(unique_texts)} texts...")

print("✅ Text embeddings pre-computed successfully!")
print(f"Embedding dimension: {len(next(iter(text_to_embedding.values())))}")

# Verify all texts in our dataset have embeddings
missing_texts = []
for _, row in df.iterrows():
    text1 = f"{row['name1']} {row['description1']}"
    text2 = f"{row['name2']} {row['description2']}"
    if text1 not in text_to_embedding:
        missing_texts.append(text1)
    if text2 not in text_to_embedding:
        missing_texts.append(text2)

if missing_texts:
    print(f"⚠️  Warning: {len(missing_texts)} texts not found in embeddings")
else:
    print("✅ All texts have corresponding embeddings")

Pre-computing all text embeddings...
This is a one-time process that will significantly speed up training...
Total unique text combinations: 1834
Computing embeddings... This may take a few minutes...
Processed 640/1834 texts...
Processed 1280/1834 texts...
✅ Text embeddings pre-computed successfully!
Embedding dimension: 384
✅ All texts have corresponding embeddings


## **5: Model 1 - Text-Only Siamese Network**

In [ ]:
class TextOnlySiameseNetwork(nn.Module):
    def __init__(self, text_dim):
        super(TextOnlySiameseNetwork, self).__init__()

        # Text processing layers
        self.text_fc = nn.Sequential(
            nn.Linear(text_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128)
        )

    def forward_one(self, text):
        # Process text
        text_out = self.text_fc(text)
        return F.normalize(text_out, p=2, dim=1)

    def forward(self, text1, text2):
        # Get embeddings for both inputs
        output1 = self.forward_one(text1)
        output2 = self.forward_one(text2)
        return output1, output2

# Initialize Model 1
model1 = TextOnlySiameseNetwork(text_dim).to(device)
print("Model 1 (Text-Only Siamese Network) created!")
print(f"Model 1 parameters: {sum(p.numel() for p in model1.parameters()):,}")

Model 1 (Text-Only Siamese Network) created!
Model 1 parameters: 361,344


## **6: Model 2 - Multi-Modal Siamese Network**

In [ ]:
class MultiModalSiameseNetwork(nn.Module):
    def __init__(self, text_dim):
        super(MultiModalSiameseNetwork, self).__init__()

        # Text processing layers
        self.text_fc = nn.Sequential(
            nn.Linear(text_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Image encoder (ResNet-50 without final classifier)
        resnet = resnet50(pretrained=True)
        self.image_encoder = nn.Sequential(*list(resnet.children())[:-1])  # Remove final FC layer

        # Freeze ResNet parameters (optional - can be unfrozen for fine-tuning)
        for param in self.image_encoder.parameters():
            param.requires_grad = False

        # Image processing layers
        self.image_fc = nn.Sequential(
            nn.Linear(2048, 512),  # ResNet-50 outputs 2048 features
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Combined feature processing
        self.combined_fc = nn.Sequential(
            nn.Linear(256 + 256, 256),  # text_dim + image_dim
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128)
        )

    def forward_one(self, text, image):
        # Process text
        text_out = self.text_fc(text)

        # Process image
        image_features = self.image_encoder(image)
        image_features = image_features.view(image_features.size(0), -1)  # Flatten
        image_out = self.image_fc(image_features)

        # Combine text and image features
        combined = torch.cat([text_out, image_out], dim=1)
        combined_out = self.combined_fc(combined)

        return F.normalize(combined_out, p=2, dim=1)

    def forward(self, text1, image1, text2, image2):
        # Get embeddings for both inputs
        output1 = self.forward_one(text1, image1)
        output2 = self.forward_one(text2, image2)
        return output1, output2

# Initialize Model 2
model2 = MultiModalSiameseNetwork(text_dim).to(device)
print("Model 2 (Multi-Modal Siamese Network) created!")
print(f"Model 2 parameters: {sum(p.numel() for p in model2.parameters()):,}")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 142MB/s]


Model 2 (Multi-Modal Siamese Network) created!
Model 2 parameters: 24,951,232


## **7: Contrastive Loss Function**

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        # Calculate Euclidean distance
        euclidean_distance = F.pairwise_distance(output1, output2, keepdim=True)

        # CORRECTED Contrastive loss calculation:
        # For a match (label=1), we want small distance (first term)
        # For a no-match (label=0), we want distance to be at least `margin` (second term)
        loss = torch.mean(
            label * torch.pow(euclidean_distance, 2) +
            (1 - label) * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2)
        )

        return loss

# Initialize loss function
criterion = ContrastiveLoss(margin=1.0)
print("✅ Contrastive Loss function initialized with corrected logic!")
print("   • label=1 (match) → minimize distance")
print("   • label=0 (no-match) → maximize distance up to margin")

✅ Contrastive Loss function initialized with corrected logic!
   • label=1 (match) → minimize distance
   • label=0 (no-match) → maximize distance up to margin


## **8: Create Data Loaders**

In [ ]:
# Create datasets with pre-computed embeddings
print("Creating datasets with pre-computed embeddings...")
train_dataset_text = ProductMatchingDataset(train_df, text_to_embedding, mode='text_only')
val_dataset_text = ProductMatchingDataset(val_df, text_to_embedding, mode='text_only')
test_dataset_text = ProductMatchingDataset(test_df, text_to_embedding, mode='text_only')

train_dataset_multimodal = ProductMatchingDataset(train_df, text_to_embedding,
                                                  image_transform, mode='multimodal')
val_dataset_multimodal = ProductMatchingDataset(val_df, text_to_embedding,
                                                image_transform, mode='multimodal')
test_dataset_multimodal = ProductMatchingDataset(test_df, text_to_embedding,
                                                 image_transform, mode='multimodal')

# Create data loaders
batch_size = 32

train_loader_text = DataLoader(train_dataset_text, batch_size=batch_size,
                              shuffle=True, num_workers=2)
val_loader_text = DataLoader(val_dataset_text, batch_size=batch_size,
                            shuffle=False, num_workers=2)
test_loader_text = DataLoader(test_dataset_text, batch_size=batch_size,
                             shuffle=False, num_workers=2)

train_loader_multimodal = DataLoader(train_dataset_multimodal, batch_size=batch_size,
                                    shuffle=True, num_workers=2)
val_loader_multimodal = DataLoader(val_dataset_multimodal, batch_size=batch_size,
                                  shuffle=False, num_workers=2)
test_loader_multimodal = DataLoader(test_dataset_multimodal, batch_size=batch_size,
                                   shuffle=False, num_workers=2)

print("✅ Data loaders created successfully with pre-computed embeddings!")
print(f"Training batches - Text: {len(train_loader_text)}, Multimodal: {len(train_loader_multimodal)}")
print("🚀 Training should now be significantly faster!")

Creating datasets with pre-computed embeddings...
✅ Data loaders created successfully with pre-computed embeddings!
Training batches - Text: 40, Multimodal: 40
🚀 Training should now be significantly faster!


## **9: Training Function**

In [ ]:
def train_model(model, train_loader, val_loader, criterion, num_epochs=10,
                learning_rate=0.001, model_type='text_only'):
    """
    Train the Siamese network model
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')

    print(f"\nTraining {model_type} model...")
    print(f"Total epochs: {num_epochs}")
    print("-" * 60)

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            optimizer.zero_grad()

            if model_type == 'text_only':
                text1 = batch['text1'].to(device)
                text2 = batch['text2'].to(device)
                labels = batch['label'].to(device)

                output1, output2 = model(text1, text2)

            else:  # multimodal
                text1 = batch['text1'].to(device)
                text2 = batch['text2'].to(device)
                image1 = batch['image1'].to(device)
                image2 = batch['image2'].to(device)
                labels = batch['label'].to(device)

                output1, output2 = model(text1, image1, text2, image2)

            loss = criterion(output1, output2, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 50 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}/{len(train_loader)}, "
                      f"Loss: {loss.item():.4f}")

        # Validation phase
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                if model_type == 'text_only':
                    text1 = batch['text1'].to(device)
                    text2 = batch['text2'].to(device)
                    labels = batch['label'].to(device)

                    output1, output2 = model(text1, text2)

                else:  # multimodal
                    text1 = batch['text1'].to(device)
                    text2 = batch['text2'].to(device)
                    image1 = batch['image1'].to(device)
                    image2 = batch['image2'].to(device)
                    labels = batch['label'].to(device)

                    output1, output2 = model(text1, image1, text2, image2)

                loss = criterion(output1, output2, labels)
                val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}, "
              f"Val Loss: {avg_val_loss:.4f}")

        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f'best_{model_type}_model.pth')

        scheduler.step()
        print("-" * 60)

    return train_losses, val_losses

## **10: Evaluation Function**

In [ ]:
def find_optimal_threshold(model, val_loader, model_type='text_only', threshold_range=(0.1, 1.5, 0.1)):
    """
    Find optimal threshold for predictions using validation set
    """
    model.eval()
    all_distances = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            if model_type == 'text_only':
                text1 = batch['text1'].to(device)
                text2 = batch['text2'].to(device)
                labels = batch['label']
                output1, output2 = model(text1, text2)
            else:  # multimodal
                text1 = batch['text1'].to(device)
                text2 = batch['text2'].to(device)
                image1 = batch['image1'].to(device)
                image2 = batch['image2'].to(device)
                labels = batch['label']
                output1, output2 = model(text1, image1, text2, image2)

            distances = F.pairwise_distance(output1, output2)
            all_distances.extend(distances.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Test different thresholds
    thresholds = np.arange(threshold_range[0], threshold_range[1], threshold_range[2])
    best_threshold = threshold_range[0]
    best_f1 = 0

    for threshold in thresholds:
        predictions = (np.array(all_distances) < threshold).astype(int)
        f1 = f1_score(all_labels, predictions)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1

def evaluate_model(model, test_loader, val_loader=None, model_type='text_only', threshold=None):
    """
    Evaluate the model on test set and calculate metrics
    """
    # Find optimal threshold if not provided
    if threshold is None and val_loader is not None:
        print(f"Finding optimal threshold for {model_type} model...")
        threshold, val_f1 = find_optimal_threshold(model, val_loader, model_type)
        print(f"Optimal threshold found: {threshold:.3f} (Val F1: {val_f1:.4f})")
    elif threshold is None:
        threshold = 0.5  # Default fallback
        print(f"Using default threshold: {threshold}")

    model.eval()
    all_distances = []
    all_labels = []
    inference_times = []

    print(f"Evaluating {model_type} model with threshold {threshold:.3f}...")

    with torch.no_grad():
        for batch in test_loader:
            start_time = time.time()

            if model_type == 'text_only':
                text1 = batch['text1'].to(device)
                text2 = batch['text2'].to(device)
                labels = batch['label']
                output1, output2 = model(text1, text2)
            else:  # multimodal
                text1 = batch['text1'].to(device)
                text2 = batch['text2'].to(device)
                image1 = batch['image1'].to(device)
                image2 = batch['image2'].to(device)
                labels = batch['label']
                output1, output2 = model(text1, image1, text2, image2)

            # Calculate distances
            distances = F.pairwise_distance(output1, output2)

            end_time = time.time()
            batch_time = (end_time - start_time) * 1000  # Convert to milliseconds
            inference_times.extend([batch_time / len(labels)] * len(labels))

            all_distances.extend(distances.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Convert distances to predictions
    all_distances = np.array(all_distances)
    all_labels = np.array(all_labels)

    # Lower distance means higher similarity (match)
    predictions = (all_distances < threshold).astype(int)

    # Calculate metrics
    precision = precision_score(all_labels, predictions)
    recall = recall_score(all_labels, predictions)
    f1 = f1_score(all_labels, predictions)
    avg_inference_time = np.mean(inference_times)

    print(f"Results for {model_type.upper()} model:")
    print(f"Threshold used: {threshold:.3f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Average Inference Time: {avg_inference_time:.2f} ms per pair")
    print("-" * 60)

    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'avg_inference_time': avg_inference_time,
        'threshold': threshold,
        'predictions': predictions,
        'distances': all_distances
    }

## **11: Train Model 1 (Text-Only)**

In [ ]:
# Train Model 1 - Text-Only Siamese Network
print("=" * 80)
print("TRAINING MODEL 1: TEXT-ONLY SIAMESE NETWORK")
print("=" * 80)

train_losses_1, val_losses_1 = train_model(
    model=model1,
    train_loader=train_loader_text,
    val_loader=val_loader_text,
    criterion=criterion,
    num_epochs=10,
    learning_rate=0.001,
    model_type='text_only'
)

print("Model 1 training completed!")

TRAINING MODEL 1: TEXT-ONLY SIAMESE NETWORK

Training text_only model...
Total epochs: 10
------------------------------------------------------------
Epoch 1/10, Batch 0/40, Loss: 0.4126
Epoch 1/10 - Train Loss: 0.2127, Val Loss: 0.5217
------------------------------------------------------------
Epoch 2/10, Batch 0/40, Loss: 0.1689
Epoch 2/10 - Train Loss: 0.2060, Val Loss: 0.5080
------------------------------------------------------------
Epoch 3/10, Batch 0/40, Loss: 0.1665
Epoch 3/10 - Train Loss: 0.1995, Val Loss: 0.5001
------------------------------------------------------------
Epoch 4/10, Batch 0/40, Loss: 0.1447
Epoch 4/10 - Train Loss: 0.2011, Val Loss: 0.5069
------------------------------------------------------------
Epoch 5/10, Batch 0/40, Loss: 0.1487
Epoch 5/10 - Train Loss: 0.2013, Val Loss: 0.5031
------------------------------------------------------------
Epoch 6/10, Batch 0/40, Loss: 0.1772
Epoch 6/10 - Train Loss: 0.1992, Val Loss: 0.5233
----------------------

## **12: Train Model 2 (Multi-Modal)**

In [ ]:
# Train Model 2 - Multi-Modal Siamese Network
print("=" * 80)
print("TRAINING MODEL 2: MULTI-MODAL SIAMESE NETWORK")
print("=" * 80)

train_losses_2, val_losses_2 = train_model(
    model=model2,
    train_loader=train_loader_multimodal,
    val_loader=val_loader_multimodal,
    criterion=criterion,
    num_epochs=10,
    learning_rate=0.001,
    model_type='multimodal'
)

print("Model 2 training completed!")

TRAINING MODEL 2: MULTI-MODAL SIAMESE NETWORK

Training multimodal model...
Total epochs: 10
------------------------------------------------------------
Error loading images at index 632: [Errno 2] No such file or directory: 'https://i5.walmartimages.com/asr/61c6f25d-d644-466c-bcf8-f1aaf8385972.8a47bbcc96eaf3c0dbd71cf3c21cd165.jpeg'
Error loading images at index 1178: [Errno 2] No such file or directory: 'https://i5.walmartimages.com/asr/20ed1491-a01e-4b94-adfe-70648c5b1e43_1.1b090523bd31293f0c31e37117aee0ed.jpeg'Error loading images at index 354: [Errno 2] No such file or directory: 'https://i5.walmartimages.com/asr/e2568d7f-93f6-4094-bdd5-ea35b3c25bc5_1.0b3cdfa147bcbb292c2c875f9c46cb1c.jpeg'
Error loading images at index 1079: [Errno 2] No such file or directory: 'https://i5.walmartimages.com/asr/b34293c0-6dbf-4e79-a7a1-237ac71a5c95.8620d491dca8051d4ea515f069252322.jpeg'
Error loading images at index 1003: [Errno 2] No such file or directory: 'https://i5.walmartimages.com/asr/9c0f27

## **13: Evaluate Both Models**

In [ ]:
# Load best models
print("Loading best trained models...")
model1.load_state_dict(torch.load('best_text_only_model.pth'))
model2.load_state_dict(torch.load('best_multimodal_model.pth'))

print("=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)

# Evaluate Model 1 with optimal threshold finding
results_1 = evaluate_model(model1, test_loader_text, val_loader_text, 'text_only')

# Evaluate Model 2 with optimal threshold finding
results_2 = evaluate_model(model2, test_loader_multimodal, val_loader_multimodal, 'multimodal')

In [ ]:
print("=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

print("\nMODEL PERFORMANCE SUMMARY:")
print("-" * 40)
print("Text-Only Siamese Network:")
print(f"  • Precision: {results_1['precision']:.4f}")
print(f"  • Recall: {results_1['recall']:.4f}")
print(f"  • F1-Score: {results_1['f1_score']:.4f}")
print(f"  • Avg Inference Time: {results_1['avg_inference_time']:.2f} ms/pair")

print("\nMulti-Modal Siamese Network:")
print(f"  • Precision: {results_2['precision']:.4f}")
print(f"  • Recall: {results_2['recall']:.4f}")
print(f"  • F1-Score: {results_2['f1_score']:.4f}")
print(f"  • Avg Inference Time: {results_2['avg_inference_time']:.2f} ms/pair")

# Determine best model based on F1-Score and inference speed
print("\n" + "="*50)
print("BEST MODEL SELECTION")
print("="*50)

# Calculate a combined score (weighted F1 and speed)
# Higher F1 is better, lower inference time is better
f1_weight = 0.7
speed_weight = 0.3

# Normalize metrics for fair comparison
max_f1 = max(results_1['f1_score'], results_2['f1_score'])
min_time = min(results_1['avg_inference_time'], results_2['avg_inference_time'])

score_1 = (f1_weight * results_1['f1_score'] / max_f1 +
           speed_weight * min_time / results_1['avg_inference_time'])

score_2 = (f1_weight * results_2['f1_score'] / max_f1 +
           speed_weight * min_time / results_2['avg_inference_time'])

print(f"\nCombined Scores (F1: {f1_weight}, Speed: {speed_weight}):")
print(f"Text-Only Model Score: {score_1:.4f}")
print(f"Multi-Modal Model Score: {score_2:.4f}")

if score_1 > score_2:
    winner = "Text-Only Siamese Network"
    winner_results = results_1
else:
    winner = "Multi-Modal Siamese Network"
    winner_results = results_2

print(f"\n🏆 WINNER: {winner}")
print(f"   Best F1-Score: {winner_results['f1_score']:.4f}")
print(f"   Inference Speed: {winner_results['avg_inference_time']:.2f} ms/pair")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)

In [ ]:
import matplotlib.pyplot as plt

# Plot training curves
plt.figure(figsize=(15, 5))

# Plot 1: Training losses
plt.subplot(1, 3, 1)
plt.plot(train_losses_1, label='Text-Only', marker='o')
plt.plot(train_losses_2, label='Multi-Modal', marker='s')
plt.title('Training Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot 2: Validation losses
plt.subplot(1, 3, 2)
plt.plot(val_losses_1, label='Text-Only', marker='o')
plt.plot(val_losses_2, label='Multi-Modal', marker='s')
plt.title('Validation Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot 3: Performance metrics
plt.subplot(1, 3, 3)
metrics = ['Precision', 'Recall', 'F1-Score']
text_scores = [results_1['precision'], results_1['recall'], results_1['f1_score']]
multi_scores = [results_2['precision'], results_2['recall'], results_2['f1_score']]

x = np.arange(len(metrics))
width = 0.35

plt.bar(x - width/2, text_scores, width, label='Text-Only', alpha=0.8)
plt.bar(x + width/2, multi_scores, width, label='Multi-Modal', alpha=0.8)

plt.xlabel('Metrics')
plt.ylabel('Score')
plt.title('Model Performance Comparison')
plt.xticks(x, metrics)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print detailed analysis
print("\nDETAILED ANALYSIS:")
print("-" * 50)
print(f"1. Model Complexity:")
print(f"   • Text-Only parameters: {sum(p.numel() for p in model1.parameters()):,}")
print(f"   • Multi-Modal parameters: {sum(p.numel() for p in model2.parameters()):,}")

print(f"\n2. Performance Trade-offs:")
if results_2['f1_score'] > results_1['f1_score']:
    print(f"   • Multi-Modal model shows {((results_2['f1_score'] - results_1['f1_score'])/results_1['f1_score']*100):.1f}% better F1-Score")
else:
    print(f"   • Text-Only model shows {((results_1['f1_score'] - results_2['f1_score'])/results_2['f1_score']*100):.1f}% better F1-Score")

speed_diff = ((results_2['avg_inference_time'] - results_1['avg_inference_time'])/results_1['avg_inference_time']*100)
if speed_diff > 0:
    print(f"   • Multi-Modal model is {speed_diff:.1f}% slower than Text-Only")
else:
    print(f"   • Multi-Modal model is {abs(speed_diff):.1f}% faster than Text-Only")

print(f"\n3. Recommendations:")
if winner == "Text-Only Siamese Network":
    print("   • Use Text-Only model for faster inference with comparable accuracy")
    print("   • Consider this model for real-time applications")
else:
    print("   • Use Multi-Modal model for better accuracy when images are available")
    print("   • Accept slower inference for improved matching precision")